# Session 4.1 -- Observability and Debugging

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Configure** Phoenix tracing for a RAG pipeline using OpenTelemetry
2. **Create** manual OTEL spans for embedding and retrieval operations
3. **Upload** a test dataset to Phoenix for structured pipeline evaluation
4. **Write** code-based evaluators that score pipeline results automatically
5. **Run** experiments comparing naive vs filtered retrieval and different models side-by-side
6. **Diagnose** pipeline failures using trace data in a production observability platform

## What You Are Working With

- `src/s4_retrieval/naive.py` -- Naive RAG pipeline (provided)
- `src/s4_retrieval/filtered.py` -- Metadata-filtered RAG (provided)
- `src/s2_embeddings/embed.py` -- Voyage AI embeddings (provided)
- `src/s3_ingestion/store.py` -- ChromaDB collection (provided)
- `src/s0_generation/generate.py` -- Claude generation (provided)

You **import and use** the pre-built modules. Today you add observability by instrumenting every stage, then run structured experiments to compare your pipeline configurations.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

# Add AI-2/ to path for src module imports (source repo layout)
_ai2 = _root / "AI-2"
if _ai2.exists():
    sys.path.insert(0, str(_ai2))

# Load .env — check both source repo (AI-2/.env) and student repo (.env at root)
from dotenv import load_dotenv
_loaded = False
for _dotenv_path in [_root / "AI-2" / ".env", _root / ".env"]:
    if _dotenv_path.exists():
        load_dotenv(_dotenv_path)
        _loaded = True
        break
if not _loaded:
    load_dotenv()  # fallback: walk up from cwd

import os
print(f"✓ Environment loaded from: {_dotenv_path if _loaded else 'default search'}")
print(f"  PHOENIX_API_KEY: {'✓ set' if os.environ.get('PHOENIX_API_KEY') else '✗ MISSING'}")
print(f"  PHOENIX_COLLECTOR_ENDPOINT: {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', '✗ MISSING')}")

In [ ]:
from phoenix.otel import register

# ╔══════════════════════════════════════════════════════════╗
# ║  CHANGE THIS to your first and last name                ║
# ║  Example: "tyler-hayes", "jane-doe", "alex-smith"       ║
# ╚══════════════════════════════════════════════════════════╝
PROJECT_NAME = "your-name-here"

# register() reads PHOENIX_COLLECTOR_ENDPOINT and PHOENIX_API_KEY
# from environment variables (loaded from .env in the previous cell)
tracer_provider = register(
    project_name=PROJECT_NAME,
    auto_instrument=True,
)

print(f"✓ Phoenix project '{PROJECT_NAME}' registered.")

In [ ]:
import json
import time
import anthropic
import phoenix as px
from phoenix.experiments import run_experiment
from opentelemetry import trace
from opentelemetry.trace import StatusCode
from openinference.instrumentation import using_attributes

from src.s2_embeddings.embed import embed_texts
from src.s3_ingestion.store import get_collection
from src.s0_generation.generate import call_claude_with_usage

import nest_asyncio
nest_asyncio.apply()

# Create a tracer for manual span creation
tracer = trace.get_tracer("rag-pipeline")

# Phoenix client for datasets and experiments
phoenix_client = px.Client()

print("✓ All imports loaded. Tracer and Phoenix client ready.")

---

## 1. First Light

Run a single API call. The Anthropic auto-instrumentor captures the Claude call automatically -- no extra code needed. After running this cell, go to Phoenix and find your trace.

In [ ]:
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=256,
    system="You are a helpful assistant. Answer in one paragraph.",
    messages=[{"role": "user", "content": "What is retrieval augmented generation?"}],
)

print(response.content[0].text)
print(f"\nTokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out")
print(f"\n→ Open Phoenix and find this trace in project: '{PROJECT_NAME}'")

### What You See in Phoenix

Click into your trace. You should see a single **LLM span** with:
- Model name (`claude-sonnet-4-5`)
- Input messages (your system prompt + user message)
- Output message (Claude's response)
- Token counts (input and output)
- Latency (how long the API call took)

This was captured **automatically** by the Anthropic instrumentor. Zero extra code.

But notice what's **missing**: when we run a full RAG query, the embedding and retrieval stages are invisible. Phoenix only sees the Claude call. Everything else is a black box.

**Your job in the next two sections: make the invisible visible.**

---

## 2. Instrument Embeddings 🟡

When you call `embed_texts()`, Phoenix doesn't see it. The embedding call is invisible -- you can't tell how long it took, what model was used, or what text was embedded.

Your job: write `traced_embed()` that wraps the Voyage embedding call in an OTEL span with kind `EMBEDDING`.

**Span attributes to set:**
- `openinference.span.kind` → `"EMBEDDING"`
- `embedding.model_name` → `"voyage-3-lite"`
- `input.value` → the text being embedded

In [ ]:
def traced_embed(texts: list[str]) -> list[list[float]]:
    """Embed texts with full OTEL tracing.

    Creates an EMBEDDING span visible in Phoenix that captures
    the model name, input text, and timing.
    """
    with tracer.start_as_current_span("embed_query") as span:
        span.set_attribute("openinference.span.kind", "EMBEDDING")

        # ── YOUR CODE HERE ──────────────────────────────────
        # 1. Set attribute "embedding.model_name" to "voyage-3-lite"
        # 2. Set attribute "input.value" to the first text in texts
        # 3. Call embed_texts(texts) to get the embeddings
        # 4. Set attribute "embedding.dimensions" to len of first embedding
        # 5. Set span status: span.set_status(StatusCode.OK)
        # 6. Return the embeddings
        # ────────────────────────────────────────────────────

        pass  # ← Remove this line when you add your code

In [ ]:
embeddings = traced_embed(["What is the vacation policy at Northbrook Partners?"])

assert embeddings is not None, "traced_embed returned None — did you forget to return the embeddings?"
assert len(embeddings) == 1, f"Expected 1 embedding, got {len(embeddings)}"
assert len(embeddings[0]) > 0, "Embedding is empty"

print(f"✓ Embedding dimensions: {len(embeddings[0])}")
print(f"\n→ Check Phoenix: you should see an EMBEDDING span in your latest trace")

---

## 3. Instrument Retrieval 🟡

Same problem -- ChromaDB queries are invisible. You can't see what was retrieved, the similarity scores, how many results came back, or whether a filter was applied.

Your job: write `traced_retrieve()` that wraps the ChromaDB query in a RETRIEVER span.

**Span attributes to set:**
- `openinference.span.kind` → `"RETRIEVER"`
- `input.value` → description of the query
- `retrieve.top_k` → number of results requested
- `retrieve.n_results` → number of results returned
- `retrieve.top_score` → highest similarity score
- `retrieve.sources` → comma-separated source names
- `retrieve.filter_applied` → the filter dict or `"none"`

In [ ]:
def traced_retrieve(
    query_embedding: list[float],
    top_k: int = 5,
    where: dict | None = None,
) -> dict:
    """Retrieve chunks from ChromaDB with full OTEL tracing.

    Creates a RETRIEVER span visible in Phoenix that captures
    result count, scores, sources, and any filters applied.

    Returns:
        dict with keys: "documents", "metadatas", "scores"
    """
    collection = get_collection()

    with tracer.start_as_current_span("retrieve_chunks") as span:
        span.set_attribute("openinference.span.kind", "RETRIEVER")

        # ── YOUR CODE HERE ──────────────────────────────────
        # 1. Build the query kwargs:
        #    - query_embeddings=[query_embedding]
        #    - n_results=top_k
        #    - include=["documents", "metadatas", "distances"]
        #    - Add where=where if where is not None
        #
        # 2. Call collection.query(**kwargs) and store result
        #
        # 3. Extract from results:
        #    - documents = results["documents"][0]
        #    - metadatas = results["metadatas"][0]
        #    - distances = results["distances"][0]
        #    - scores = [round(1 - d, 4) for d in distances]
        #    - sources = [m.get("source", "unknown") for m in metadatas]
        #
        # 4. Set span attributes:
        #    - "retrieve.top_k" → top_k
        #    - "retrieve.n_results" → len(documents)
        #    - "retrieve.top_score" → float(scores[0]) if scores else 0.0
        #    - "retrieve.sources" → ", ".join(sources)
        #    - "retrieve.filter_applied" → str(where) if where else "none"
        #
        # 5. Set span status: span.set_status(StatusCode.OK)
        #
        # 6. Return {"documents": documents, "metadatas": metadatas, "scores": scores}
        # ────────────────────────────────────────────────────

        pass  # ← Remove this line when you add your code

In [ ]:
embedding = traced_embed(["What is the vacation policy at Northbrook Partners?"])
results = traced_retrieve(embedding[0], top_k=5)

assert results is not None, "traced_retrieve returned None"
assert "documents" in results, "Missing 'documents' key in results"
assert "scores" in results, "Missing 'scores' key in results"
assert len(results["documents"]) > 0, "No documents retrieved"

print(f"✓ Retrieved {len(results['documents'])} chunks")
print(f"  Top score: {results['scores'][0]:.4f}")
print(f"  Sources: {[m.get('source', '?') for m in results['metadatas']]}")
print(f"\n→ Check Phoenix: your trace should now show EMBEDDING + RETRIEVER spans")

### Full Traced Pipeline

Now we combine your `traced_embed()` and `traced_retrieve()` with the auto-instrumented Claude call into a complete traced RAG query. This function is provided -- it uses YOUR implementations.

In [ ]:
def traced_rag_query(question: str, use_filter: dict | None = None, top_k: int = 5, model: str = "claude-sonnet-4-5") -> dict:
    """Run a full RAG query with tracing on every stage.

    Uses YOUR traced_embed() and traced_retrieve() implementations.
    The Claude API call is auto-instrumented by Phoenix.
    """
    with tracer.start_as_current_span("rag_pipeline") as parent:
        parent.set_attribute("openinference.span.kind", "CHAIN")
        parent.set_attribute("input.value", question)

        # Stage 1: Embed (your traced function)
        query_embedding = traced_embed([question])[0]

        # Stage 2: Retrieve (your traced function)
        results = traced_retrieve(query_embedding, top_k=top_k, where=use_filter)

        # Stage 3: Generate (auto-instrumented)
        context_blocks = []
        for doc, meta, score in zip(results["documents"], results["metadatas"], results["scores"]):
            source = meta.get("source", "Unknown")
            context_blocks.append(f"[Source: {source}, Score: {score:.3f}]\n{doc}")
        context = "\n\n---\n\n".join(context_blocks)

        system_prompt = (
            "You are a Q&A assistant for Northbrook Partners. "
            "Answer using ONLY the provided context. Cite sources by name. "
            "If the context is insufficient, say: 'I don't have enough information to answer that question.'"
        )

        client = anthropic.Anthropic()
        response = client.messages.create(
            model=model,
            max_tokens=512,
            system=system_prompt,
            messages=[{"role": "user", "content": f"Context:\n\n{context}\n\n---\n\nQuestion: {question}"}],
            temperature=0.0,
        )

        answer = response.content[0].text
        parent.set_attribute("output.value", answer[:500])
        parent.set_status(StatusCode.OK)

        return {
            "answer": answer,
            "sources": results["metadatas"],
            "scores": results["scores"],
            "tokens": {
                "input": response.usage.input_tokens,
                "output": response.usage.output_tokens,
            },
        }

print("✓ traced_rag_query() defined — uses YOUR traced_embed() and traced_retrieve().")

In [ ]:
result = traced_rag_query("What is the vacation policy at Northbrook Partners?")

print("=== Answer ===")
print(result["answer"][:500])
print(f"\nScores: {[f'{s:.3f}' for s in result['scores']]}")
print(f"Tokens: {result['tokens']['input']} in / {result['tokens']['output']} out")
print(f"\n→ Check Phoenix: full trace with CHAIN → EMBEDDING → RETRIEVER → LLM")

In [ ]:
# ── Intentional Failures: See what bad traces look like ────────────
# These queries are DESIGNED to fail so you can see failure patterns in Phoenix.

print("Running failure queries...\n")

# Failure 1: Topic not in corpus — should get refusal or hallucination
print("1. Query about something NOT in the corpus:")
result = traced_rag_query("What is the company's cryptocurrency investment policy?")
print(f"   Answer: {result['answer'][:150]}...")
print(f"   Top score: {result['scores'][0]:.3f} (low = bad retrieval)")
print(f"   Status in Phoenix: GREEN ✓ — the pipeline ran fine, the results are just bad\n")

# Failure 2: Wrong filter — financial question filtered to memos
print("2. Financial question with WRONG filter (memo instead of financial):")
result = traced_rag_query("What were the Q3 revenue figures?", use_filter={"doc_type": "memo"})
print(f"   Answer: {result['answer'][:150]}...")
print(f"   Sources: {[m.get('source', '?') for m in result['sources']]}")
print(f"   Status in Phoenix: GREEN ✓ — wrong sources, but no crash\n")

# Failure 3: Vague query — everything scores low
print("3. Vague query with no specificity:")
result = traced_rag_query("Tell me about the stuff")
print(f"   Answer: {result['answer'][:150]}...")
print(f"   Scores: {[f'{s:.3f}' for s in result['scores']]}")
print(f"   Status in Phoenix: GREEN ✓ — all scores low, but the code worked\n")

# Failure 4: Genuine ERROR — invalid filter SYNTAX crashes ChromaDB
# This creates a RED error trace in Phoenix
print("4. Invalid filter syntax — this will CRASH:")
from opentelemetry.trace import Status
with tracer.start_as_current_span("rag_pipeline") as span:
    span.set_attribute("openinference.span.kind", "CHAIN")
    span.set_attribute("input.value", "What is the vacation policy? [invalid filter]")
    try:
        embedding = traced_embed(["What is the vacation policy?"])[0]
        # This filter uses an invalid operator — ChromaDB rejects it
        traced_retrieve(embedding, where={"doc_type": {"$invalid_op": "financial"}})
        span.set_status(StatusCode.OK)
    except Exception as e:
        span.set_status(Status(StatusCode.ERROR, str(e)))
        span.record_exception(e)
        print(f"   Error: {type(e).__name__}: {e}")
        print(f"   Status in Phoenix: RED ✗ — an actual infrastructure crash")

print(f"\n→ Check Phoenix: compare green ✓ traces (1-3) vs the red ✗ trace (4)")
print(f"   1-3 ran fine but produced bad results — you need EVALUATORS to catch those")
print(f"   4 actually crashed — SPAN STATUS catches that")

---

## 4. Metadata, Datasets & Experiments

Your traces work. Now let's use Phoenix's higher-level features:

1. **`using_attributes()`** — attach metadata and tags to individual traces
2. **Datasets** — upload a suite of test queries to Phoenix for structured evaluation
3. **Experiments** — run your pipeline against the dataset, compare configurations side-by-side
4. **Evaluators** — write code that automatically scores each result

This is the difference between "I looked at a few traces" and "I systematically evaluated my pipeline."

In [ ]:
# Quick demo: tag a query with metadata
with using_attributes(
    session_id="demo",
    user_id=PROJECT_NAME,
    metadata={"retriever_type": "naive", "demo": True},
    tags=["demo"],
):
    result = traced_rag_query("How does the Navigator onboarding program work?")

print(f"Answer: {result['answer'][:200]}...")
print(f"\n→ Check Phoenix: find this trace and look for the metadata and tags")

### From Manual to Systematic: Datasets & Experiments

`using_attributes()` works for tagging individual queries. But to **systematically compare** your naive vs filtered pipeline across many queries, you need something more structured.

**Phoenix Datasets & Experiments:**

| Concept | What it does |
|---------|-------------|
| **Dataset** | A set of test queries with expected outcomes, uploaded to Phoenix |
| **Task Function** | Wraps your pipeline so Phoenix can call it on each test query |
| **Evaluators** | Code that automatically scores each result (pass/fail) |
| **Experiment** | Runs the task function on every dataset example, records results + scores |

We've provided `test_queries.json` with 10 curated queries spanning all document types. Your job: write the task function and evaluators, then run three experiments to compare configurations.

In [ ]:
# Load pre-built test queries
with open("test_queries.json") as f:
    test_queries = json.load(f)

print(f"Loaded {len(test_queries)} test queries:\n")
for i, q in enumerate(test_queries):
    print(f"  {i+1:>2}. [{q['expected_doc_type']:>10}] {q['query']}")

# Upload to Phoenix as a dataset (or load existing)
# Version changes when test queries are updated
DATASET_VERSION = "v2"
dataset_name = f"rag-eval-{PROJECT_NAME}-{DATASET_VERSION}"
try:
    dataset = phoenix_client.upload_dataset(
        dataset_name=dataset_name,
        inputs=[{"query": q["query"]} for q in test_queries],
        outputs=[{
            "expected_source": q["expected_source"],
            "expected_doc_type": q["expected_doc_type"],
        } for q in test_queries],
    )
    print(f"\n✓ Dataset '{dataset_name}' created in Phoenix")
except Exception:
    dataset = phoenix_client.get_dataset(name=dataset_name)
    print(f"\n✓ Dataset '{dataset_name}' already exists — using existing version")

print(f"→ Check Phoenix: Datasets tab → you should see your {len(test_queries)} test queries")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🟡 YOUR CODE: Write the naive RAG task function
# ══════════════════════════════════════════════════════════════
#
# Phoenix calls this function ONCE per test query in the dataset.
# It receives an Example object with:
#   - example.input["query"]            → the test query string
#   - example.output["expected_source"]   → the source that should be retrieved
#   - example.output["expected_doc_type"] → the doc_type for filtering
#
# Your function should:
#   1. Get the query from example.input["query"]
#   2. Call traced_rag_query(query) with NO filter (naive retrieval)
#   3. Return a dict with the keys shown below


def naive_task(example):
    """Run a query through the naive (unfiltered) RAG pipeline."""
    # ── YOUR CODE HERE ──────────────────────────────────
    # query = ...
    # result = traced_rag_query(...)
    # return {
    #     "answer": result["answer"],
    #     "sources": [m.get("source", "") for m in result["sources"]],
    #     "scores": result["scores"],
    #     "top_score": result["scores"][0] if result["scores"] else 0.0,
    #     "tokens": result["tokens"],
    # }
    # ────────────────────────────────────────────────────
    pass  # ← Remove this and add your code


# These two are provided — study how they differ from naive_task

def filtered_task(example):
    """Run a query through the filtered RAG pipeline."""
    query = example.input["query"]
    doc_type = example.output["expected_doc_type"]
    result = traced_rag_query(query, use_filter={"doc_type": doc_type})
    return {
        "answer": result["answer"],
        "sources": [m.get("source", "") for m in result["sources"]],
        "scores": result["scores"],
        "top_score": result["scores"][0] if result["scores"] else 0.0,
        "tokens": result["tokens"],
    }


def haiku_task(example):
    """Run a query through filtered RAG with claude-haiku-4-5 (cheaper model)."""
    query = example.input["query"]
    doc_type = example.output["expected_doc_type"]
    result = traced_rag_query(query, use_filter={"doc_type": doc_type}, model="claude-haiku-4-5")
    return {
        "answer": result["answer"],
        "sources": [m.get("source", "") for m in result["sources"]],
        "scores": result["scores"],
        "top_score": result["scores"][0] if result["scores"] else 0.0,
        "tokens": result["tokens"],
    }


print("✓ Task functions defined: naive_task, filtered_task, haiku_task")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🟡 YOUR CODE: Write two evaluators
# ══════════════════════════════════════════════════════════════
#
# Evaluators are functions Phoenix calls after each experiment run.
# They receive:
#   - output: the dict your task function returned
#   - expected: the outputs dict from the dataset (expected_source, expected_doc_type)
#
# Return True/False (pass/fail) or a float (0.0 to 1.0) for scoring.
#
# IMPORTANT: Parameter names matter! Phoenix injects by name.
# Use exactly: output, expected

# ── Doc type classifier (used by evaluators) ────────────────
DOC_TYPE_RULES = {
    "policy": ["policy", "handbook"],
    "financial": ["financial"],
    "meeting": ["meeting", "standup", "committee"],
    "memo": ["memo"],
}

def _source_matches_type(source: str, doc_type: str) -> bool:
    """Check if a source filename matches a doc_type."""
    keywords = DOC_TYPE_RULES.get(doc_type, [])
    return any(kw in source.lower() for kw in keywords)


def eval_correct_source(output, expected) -> bool:
    """Did the expected source appear in the retrieved results?"""
    # ── YOUR CODE HERE ──────────────────────────────────
    # 1. Get expected_source from the expected dict
    # 2. Get the list of sources from the output dict
    # 3. Return True if expected_source is in sources
    # ────────────────────────────────────────────────────
    pass  # ← Remove this and add your code


def eval_source_relevance(output, expected) -> float:
    """What fraction of retrieved sources are from the expected doc type?

    Returns 0.0 to 1.0. This is where filtered retrieval shines —
    all results are from the correct type, while naive returns mixed results.
    """
    # ── YOUR CODE HERE ──────────────────────────────────
    # 1. Get expected_doc_type from expected dict
    # 2. Get list of sources from output dict
    # 3. Count how many sources match the expected doc type
    #    (use _source_matches_type helper)
    # 4. Return count / total as a float
    # ────────────────────────────────────────────────────
    pass  # ← Remove this and add your code


def eval_score_above_threshold(output, expected) -> bool:
    """Was the top retrieval score above 0.5?"""
    # ── YOUR CODE HERE ──────────────────────────────────
    # 1. Get top_score from output dict (default to 0.0)
    # 2. Return True if top_score > 0.5
    # ────────────────────────────────────────────────────
    pass  # ← Remove this and add your code


def eval_has_answer(output, expected) -> bool:
    """Did the pipeline produce a real answer (not a refusal)?"""
    answer = output.get("answer", "")
    return bool(answer.strip()) and "don't have enough information" not in answer.lower()


print("✓ Evaluators defined: eval_correct_source, eval_source_relevance, eval_score_above_threshold, eval_has_answer")

In [ ]:
# ── Run Three Experiments ──────────────────────────────────────────

evaluators = {
    "correct_source": eval_correct_source,
    "source_relevance": eval_source_relevance,
    "score_threshold": eval_score_above_threshold,
    "has_answer": eval_has_answer,
}

# Experiment 1: Naive Retrieval (no filter, Sonnet)
print("Running Experiment 1: Naive Retrieval...")
exp_naive = run_experiment(
    dataset=dataset,
    task=naive_task,
    evaluators=evaluators,
    experiment_name=f"naive / {PROJECT_NAME}",
)
print("✓ Naive experiment complete\n")

# Experiment 2: Filtered Retrieval (doc_type filter, Sonnet)
print("Running Experiment 2: Filtered Retrieval...")
exp_filtered = run_experiment(
    dataset=dataset,
    task=filtered_task,
    evaluators=evaluators,
    experiment_name=f"filtered / {PROJECT_NAME}",
)
print("✓ Filtered experiment complete\n")

# Experiment 3: Filtered + Haiku (same filter, cheaper model)
print("Running Experiment 3: Filtered + Haiku...")
exp_haiku = run_experiment(
    dataset=dataset,
    task=haiku_task,
    evaluators=evaluators,
    experiment_name=f"filtered-haiku / {PROJECT_NAME}",
)
print("✓ All 3 experiments complete!")
print(f"\n→ Open Phoenix → Datasets → '{dataset_name}' → Experiments tab")
print("  You should see all 3 experiments with evaluator scores side-by-side")

### What You See in Phoenix

Navigate to **Datasets** → your dataset → **Experiments** tab.

You should see three experiments side-by-side:

| Experiment | Retrieval | Model | What it tests |
|-----------|-----------|-------|--------------|
| naive | No filter | Sonnet 4.5 | Baseline — searches everything |
| filtered | doc_type filter | Sonnet 4.5 | Does filtering improve source accuracy? |
| filtered-haiku | doc_type filter | Haiku 4.5 | Same quality at lower cost? |

**Columns to compare:**
- **correct_source** — Did the right document get retrieved?
- **score_threshold** — Was the retrieval confident?
- **has_answer** — Did the model answer (vs refuse)?
- **Metrics toggle** (top-right) — Cost, latency, token counts per experiment

Click into any result row to see the full trace — all the way down to individual spans.

---

## 5. Experiment Analysis 🟡

You now have three experiments in Phoenix with evaluator scores across 10 queries. Use the Phoenix UI to answer these questions.

Navigate to **Datasets** → your dataset → **Experiments** tab → compare all three.

### Question 1: Evaluator Pass Rates

Compare the **correct_source** pass rate across your three experiments. Which retrieval strategy retrieved the expected source most often? By how much?

**Your answer:**

*(Write here)*

### Question 2: Shared Failures

Find a query where both naive AND filtered retrieval failed the **correct_source** evaluator. What was the query? Why do you think both strategies missed?

**Your answer:**

*(Write here)*

### Question 3: Sonnet vs Haiku

Compare your **filtered** (Sonnet) and **filtered-haiku** experiments. Use the Metrics toggle to check cost and latency. How do they differ? Did quality (evaluator scores) change?

**Your answer:**

*(Write here)*

### Question 4: Score Paradox

Click into individual experiment results. Find a query where naive retrieval had a HIGHER top score than filtered, but filtered still produced a better answer. What explains this?

**Your answer:**

*(Write here)*

### Question 5: Cost Projection

Using the cost and token data from your experiments, estimate the monthly cost of running 500 queries per day for 30 days with each configuration:
- Naive + Sonnet
- Filtered + Sonnet
- Filtered + Haiku

Which configuration would you recommend for production? Why?

**Your answer:**

*(Write here)*

---

## 6. The Whodunit Challenge

Your instructor will walk through three failure traces on the projector. For each case: **which span reveals the problem? What is the root cause? What would you change?**

The cases below are for your reference.

In [ ]:
case_1 = {
    "query": "What are the expense reimbursement procedures?",
    "spans": {
        "embed": {"latency_ms": 1847, "model": "voyage-3-large", "input_chars": 48},
        "retrieve": {
            "latency_ms": 45, "n_results": 5,
            "top_score": 0.31, "score_spread": 0.13,
            "sources": ["meeting_notes_q3.md", "employee_handbook.md", "standup_notes_jan.md", "security_update.md", "meeting_notes_q3.md"],
            "filter_applied": "none",
        },
        "generate": {"latency_ms": 1523, "model": "claude-sonnet-4-5", "input_tokens": 2341, "output_tokens": 187},
    },
    "total_latency_ms": 3415,
}
print("CASE 1: The Slow Embed")
print(json.dumps(case_1, indent=2))

In [ ]:
case_2 = {
    "query": "Tell me about the stuff",
    "spans": {
        "embed": {"latency_ms": 98, "model": "voyage-3-lite", "input_chars": 24},
        "retrieve": {
            "latency_ms": 22, "n_results": 5,
            "top_score": 0.28, "score_spread": 0.07,
            "sources": ["financial_report_q3.md", "employee_handbook.md", "meeting_notes_q3.md", "standup_notes_jan.md", "remote_work_policy.md"],
            "filter_applied": "none",
        },
        "generate": {"latency_ms": 1102, "model": "claude-sonnet-4-5", "input_tokens": 2156, "output_tokens": 94},
    },
    "total_latency_ms": 1222,
}
print("CASE 2: The Vague Query")
print(json.dumps(case_2, indent=2))

In [ ]:
case_3 = {
    "query": "What is the maximum PTO carryover for senior employees?",
    "spans": {
        "embed": {"latency_ms": 105, "model": "voyage-3-lite", "input_chars": 55},
        "retrieve": {
            "latency_ms": 19, "n_results": 5,
            "top_score": 0.87, "score_spread": 0.13,
            "sources": ["vacation_policy_2025.md", "vacation_policy_2025.md", "employee_handbook.md", "vacation_policy_2025.md", "hr_policy.md"],
            "filter_applied": "none",
        },
        "generate": {"latency_ms": 1876, "model": "claude-sonnet-4-5", "input_tokens": 2934, "output_tokens": 267},
    },
    "total_latency_ms": 2000,
    "generated_answer": "Senior employees with 10+ years can carry over up to 10 days of PTO.",
    "note": "The retrieved chunks discuss general PTO accrual rates but do NOT mention seniority-based carryover limits.",
}
print("CASE 3: The Confidence Mirage")
display_case = {k: v for k, v in case_3.items() if k != "note"}
print(json.dumps(display_case, indent=2))
print()
print("The generated answer was:")
print(f'  "{case_3["generated_answer"]}"')
print()
print("The retrieved chunks discuss general PTO accrual rates but do NOT mention")
print("seniority-based carryover limits anywhere.")

### Whodunit -- Answer Key

*(Your instructor will discuss these after the exercise.)*

---

## 7. Session Recap

### What We Built Today

1. **Phoenix project** — connected to a production tracing platform
2. **Embedding spans** — manual OTEL instrumentation for Voyage calls
3. **Retrieval spans** — manual OTEL instrumentation for ChromaDB queries
4. **Test dataset** — 10 curated queries uploaded to Phoenix
5. **Code-based evaluators** — correct_source, score_threshold, has_answer
6. **Three experiments** — naive vs filtered vs model comparison, side-by-side
7. **Diagnostic workflow** — the Whodunit retrieval autopsy

### Key Takeaways

- **Structured tracing is not optional.** In non-deterministic systems, you must log what happened because you cannot reason about it after the fact.
- **Most RAG failures are retrieval failures.** If the right chunks don't reach the model, no prompt engineering saves you.
- **Experiments beat eyeballing.** Running the same queries through different configurations with evaluators gives you data, not vibes.
- **Cost is a metric too.** Haiku vs Sonnet isn't just about quality — it's about budget. Tracing makes that visible.
- **Production tools exist.** Phoenix, LangSmith, Datadog — you don't have to build logging from scratch.

### What You've Built Across AI-2

| Session | What We Built | Status |
|---------|--------------|--------|
| 1.1 | API connection + generation | ✓ |
| 1.2 | Structured extraction with Pydantic | ✓ |
| 2.1 | Embeddings + similarity measurement | ✓ |
| 2.2 | Chunking + vector store ingestion | ✓ |
| 3.1 | Naive RAG pipeline | ✓ |
| 3.2 | Metadata-filtered RAG + evaluation | ✓ |
| **4.1** | **Observability + Phoenix experiments** | **✓ (today)** |
| 4.2 | Module Test | Next session |

### Session 4.2 — Module Test

1. **Submit Lab 2** at the start of class
2. **Written Exam** — 50 minutes, topics from all sessions

### AI-3 Preview

Your Phoenix traces follow you into AI-3:
- **Sessions** → Chat state management (AI-3 1.1)
- **Annotations** → Feedback collection (AI-3 2.2)
- **Evaluations** → Eval-Driven Development (AI-3 3.2)
- **Datasets** → Golden test sets (AI-3 3.2)